# Fine-tuning DistilCamemBERT sur les avis Ynov

Ce notebook fine-tune `cmarkea/distilcamembert-base` sur le dataset binaire `avis_ynov_augmented.csv` (197 positifs / 197 négatifs), évalue les performances et pousse le modèle final sur Hugging Face Hub.

**Plateforme cible** : Google Colab (T4 GPU gratuit)  
**Durée estimée** : 5-10 minutes pour l'entraînement complet  
**Sortie** : modèle pushed sur `Ahmat293/distilcamembert-ynov-augmented`

## 1. Installation des dépendances

`transformers`, `datasets` et `evaluate` ne sont pas pré-installés sur Colab. `accelerate` est requis par `Trainer`. `huggingface_hub` permettra le push final.

In [ ]:
!pip install -q transformers datasets evaluate accelerate huggingface_hub scikit-learn

## 2. Authentification Hugging Face

Pour pouvoir pusher le modèle final, on s'authentifie avec un token HF.  
Créer un token avec **write access** sur https://huggingface.co/settings/tokens si besoin.

In [ ]:
from huggingface_hub import notebook_login
notebook_login()

## 3. Upload du dataset

Upload `avis_ynov_augmented.csv` (généré localement par data augmentation, 394 lignes binaires).

In [ ]:
from google.colab import files
uploaded = files.upload()  # selectionner avis_ynov_augmented.csv
csv_path = "avis_ynov_augmented.csv"

## 4. Chargement et split stratifié

On lit le CSV, on encode les labels, et on splitte en 70% train / 15% validation / 15% test.
La stratification se fait sur `sentiment_label × source` pour garantir une représentation équilibrée des avis réels et synthétiques dans chaque split.

In [ ]:
import pandas as pd
from sklearn.model_selection import train_test_split

df = pd.read_csv(csv_path)
print(f"Total : {len(df)} lignes")
print(f"Sentiment : {df['sentiment_label'].value_counts().to_dict()}")
print(f"Source : {df['source'].value_counts().to_dict()}")

label2id = {"negatif": 0, "positif": 1}
id2label = {0: "negatif", 1: "positif"}
df["label"] = df["sentiment_label"].map(label2id)

df["strat"] = df["sentiment_label"] + "_" + df["source"]

train_val, test = train_test_split(
    df, test_size=0.15, stratify=df["strat"], random_state=42
)
train, val = train_test_split(
    train_val, test_size=0.176,
    stratify=train_val["strat"], random_state=42
)

print(f"\nTrain : {len(train)}, Val : {len(val)}, Test : {len(test)}")
print(f"Train sentiment : {train['sentiment_label'].value_counts().to_dict()}")
print(f"Test  sentiment : {test['sentiment_label'].value_counts().to_dict()}")


## 5. Tokenization

On tokenise les commentaires avec le tokenizer de DistilCamemBERT (`max_length=512`, troncature à droite).  
Conversion pandas → `datasets.Dataset` pour interfaçage avec `Trainer`.

In [ ]:
from datasets import Dataset
from transformers import AutoTokenizer

MODEL_BASE = "camembert-base"
tokenizer = AutoTokenizer.from_pretrained(MODEL_BASE)

def tokenize(batch):
    return tokenizer(
        batch["comment"],
        padding="max_length",
        truncation=True,
        max_length=512,
    )

train_ds = Dataset.from_pandas(train[["comment", "label", "source"]]).map(tokenize, batched=True)
val_ds   = Dataset.from_pandas(val[["comment", "label", "source"]]).map(tokenize, batched=True)
test_ds  = Dataset.from_pandas(test[["comment", "label", "source"]]).map(tokenize, batched=True)

print(f"Train : {len(train_ds)}, Val : {len(val_ds)}, Test : {len(test_ds)}")
print(f"Colonnes : {train_ds.column_names}")


## 6. Modèle, métriques et configuration d'entraînement

Modèle DistilCamemBERT avec tête de classification 2 classes.  
`compute_metrics` calcule accuracy + F1 macro + F1 par classe.  
Hyperparamètres selon la spec : 5 epochs, lr=2e-5, batch=16, weight_decay=0.01, warmup=0.1.

In [ ]:
import numpy as np
from transformers import AutoModelForSequenceClassification, TrainingArguments, Trainer
from sklearn.metrics import accuracy_score, f1_score, classification_report

model = AutoModelForSequenceClassification.from_pretrained(
    MODEL_BASE,
    num_labels=2,
    id2label=id2label,
    label2id=label2id,
)

def compute_metrics(eval_pred):
    logits, labels = eval_pred
    preds = np.argmax(logits, axis=-1)
    return {
        "accuracy": accuracy_score(labels, preds),
        "f1_macro": f1_score(labels, preds, average="macro"),
        "f1_negatif": f1_score(labels, preds, pos_label=0, average="binary"),
        "f1_positif": f1_score(labels, preds, pos_label=1, average="binary"),
    }

training_args = TrainingArguments(
    output_dir="./distilcamembert-ynov-augmented",
    num_train_epochs=5,
    per_device_train_batch_size=16,
    per_device_eval_batch_size=32,
    learning_rate=2e-5,
    weight_decay=0.01,
    warmup_ratio=0.1,
    eval_strategy="epoch",
    save_strategy="epoch",
    load_best_model_at_end=True,
    metric_for_best_model="f1_macro",
    greater_is_better=True,
    logging_steps=10,
    push_to_hub=False,
    report_to="none",
)

trainer = Trainer(
    model=model,
    args=training_args,
    train_dataset=train_ds,
    eval_dataset=val_ds,
    processing_class=tokenizer,
    compute_metrics=compute_metrics,
)


## 7. Entraînement

5 epochs sur ~276 exemples train, ~6 minutes sur T4 gratuit.  
Le best model selon F1 macro sur la validation est rechargé automatiquement à la fin.

In [ ]:
trainer.train()

## 8. Évaluation sur le test set

Trois évaluations :
1. **Globale** sur tout le test set
2. **Subset `real`** uniquement
3. **Subset `synthetic`** uniquement

Si l'écart entre real et synthetic est > 10 points de F1, c'est un signal de biais de génération.

In [ ]:
import json as _json

def evaluate_subset(name, dataset):
    metrics = trainer.evaluate(eval_dataset=dataset)
    print(f"\n=== {name} (n={len(dataset)}) ===")
    for k, v in metrics.items():
        if k.startswith("eval_") and k != "eval_runtime" and "samples" not in k and "steps" not in k:
            if isinstance(v, (int, float)):
                print(f"  {k}: {v:.4f}")
    return metrics

global_metrics = evaluate_subset("Global test set", test_ds)

real_ds      = test_ds.filter(lambda ex: ex["source"] == "real")
synthetic_ds = test_ds.filter(lambda ex: ex["source"] == "synthetic")

real_metrics      = evaluate_subset("Test set REAL", real_ds)
synthetic_metrics = evaluate_subset("Test set SYNTHETIC", synthetic_ds) if len(synthetic_ds) > 0 else {"eval_f1_macro": float("nan")}

all_metrics = {
    "global":    {k: float(v) for k, v in global_metrics.items() if isinstance(v, (int, float))},
    "real":      {k: float(v) for k, v in real_metrics.items()      if isinstance(v, (int, float))},
    "synthetic": {k: float(v) for k, v in synthetic_metrics.items() if isinstance(v, (int, float))},
}
with open("eval_results.json", "w") as f:
    _json.dump(all_metrics, f, indent=2)

print("\n=== Synthese ===")
print(f"Global F1 macro     : {global_metrics['eval_f1_macro']:.4f}")
print(f"Real F1 macro       : {real_metrics['eval_f1_macro']:.4f}")
if len(synthetic_ds) > 0:
    print(f"Synthetic F1 macro  : {synthetic_metrics['eval_f1_macro']:.4f}")
    gap = abs(real_metrics["eval_f1_macro"] - synthetic_metrics["eval_f1_macro"])
    print(f"Ecart real/synthetic: {gap:.4f}")


## 8b. Matrice de confusion sur le test global

In [ ]:
import matplotlib.pyplot as plt
from sklearn.metrics import confusion_matrix, ConfusionMatrixDisplay

predictions = trainer.predict(test_ds)
preds = np.argmax(predictions.predictions, axis=-1)
labels = predictions.label_ids

cm = confusion_matrix(labels, preds)
disp = ConfusionMatrixDisplay(cm, display_labels=["negatif", "positif"])
fig, ax = plt.subplots(figsize=(5, 4))
disp.plot(ax=ax, cmap="Blues", values_format="d")
plt.title("Matrice de confusion - DistilCamemBERT Ynov augmente")
plt.tight_layout()
plt.show()

print(classification_report(labels, preds, target_names=["negatif", "positif"]))


## 9. Push sur Hugging Face Hub

Push du modèle final vers `Ahmat293/distilcamembert-ynov-augmented`.  
Le repo est créé automatiquement s'il n'existe pas. Visibilité par défaut : public.

In [ ]:
HF_REPO = "Ahmat293/distilcamembert-ynov-augmented"

trainer.push_to_hub(
    HF_REPO,
    commit_message="DistilCamemBERT fine-tune sur avis Ynov augmentes (binaire)",
)
tokenizer.push_to_hub(HF_REPO)

print(f"\n[OK] Modele disponible sur https://huggingface.co/{HF_REPO}")
